** Lab 5 Exercise 1 By Prajakta Ghorpade SCFP125004**

---
Exercise 1 — Customer Review to Support Ticket Pipeline
Build a 2-step pipeline. Step 1 extracts the core complaint, product/feature mentioned, and customer sentiment from a raw customer review as structured data. Step 2 generates a support ticket summary for the internal team, using only the structured data from Step 1, not the raw review text.

Input: review_text

Output: the structured complaint data from Step 1, and the support ticket summary from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: complaint, product_or_feature, sentiment)

Step 2 — Template: ChatPromptTemplate (system persona: "support ticket writer")  |  Parser: StrOutputParser


Step 1 — Install the required libraries

In [1]:
%pip install -q -U langchain-groq langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.6 MB/s eta 0:00:00


Step 2 — Set up the language model

In [8]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    temperature=0
)

print("Language model configured.")

Language model configured.


In [9]:
response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


Step 3 — Define the structured output

In [3]:
from pydantic import BaseModel, Field

class ComplaintData(BaseModel):
    complaint: str = Field(description="Main customer problem")
    product_or_feature: str = Field(description="Product or feature mentioned")
    sentiment: str = Field(description="Customer sentiment")

Step 4 — Create the first prompt and parser

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser1 = PydanticOutputParser(
    pydantic_object=ComplaintData
)

prompt1 = PromptTemplate(
    template="""
Extract the following information from the customer review:
1. complaint
2. product_or_feature
3. sentiment

Use only the information in the review.
Do not invent details.

{format_instructions}

Customer review:
{review_text}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser1.get_format_instructions()
    }
)

print("Step 1 prompt and parser are ready.")

Step 1 prompt and parser are ready.


Step 5 — Run Step 1 and extract the complaint

In [10]:
review_text = (
    "My headphones stopped working after two days. "
    "I am very disappointed with the product."
)

step1_chain = prompt1 | llm | parser1

complaint_data = step1_chain.invoke(
    {"review_text": review_text}
)

print("STEP 1: STRUCTURED COMPLAINT DATA")
print(complaint_data)
print(complaint_data.model_dump())

STEP 1: STRUCTURED COMPLAINT DATA
complaint='Headphones stopped working after two days' product_or_feature='headphones' sentiment='negative'
{'complaint': 'Headphones stopped working after two days', 'product_or_feature': 'headphones', 'sentiment': 'negative'}


Step 6 — Create the second prompt

In [11]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a support ticket writer. "
        "Write clear, concise internal support tickets. "
        "Use only the supplied structured data. "
        "Do not invent information."
    ),
    (
        "human",
        """Create a support ticket summary using these details:

Complaint: {complaint}
Product or feature: {product_or_feature}
Customer sentiment: {sentiment}

Include the issue, customer sentiment, and a suitable
next action for the internal support team."""
    )
])

print("Step 2 chat prompt is ready.")

Step 2 chat prompt is ready.


Step 7 — Create the second parser and generate the ticket

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser2 = StrOutputParser()

step2_chain = chat_prompt | llm | parser2

ticket_summary = step2_chain.invoke(
    complaint_data.model_dump()
)

print("STEP 2: SUPPORT TICKET SUMMARY")
print(ticket_summary)

STEP 2: SUPPORT TICKET SUMMARY
**Support Ticket Summary**

- **Issue:** Headphones stopped working after two days of use.  
- **Customer Sentiment:** Negative.  
- **Next Action:** Escalate to the hardware repair team, initiate a replacement request, and schedule a return shipment for the defective unit.


Step 8 — Display the final result neatly

In [13]:
print("CUSTOMER REVIEW")
print(review_text)

print("\nSTEP 1: STRUCTURED COMPLAINT DATA")
print(complaint_data.model_dump())

print("\nSTEP 2: SUPPORT TICKET SUMMARY")
print(ticket_summary)

CUSTOMER REVIEW
My headphones stopped working after two days. I am very disappointed with the product.

STEP 1: STRUCTURED COMPLAINT DATA
{'complaint': 'Headphones stopped working after two days', 'product_or_feature': 'headphones', 'sentiment': 'negative'}

STEP 2: SUPPORT TICKET SUMMARY
**Support Ticket Summary**

- **Issue:** Headphones stopped working after two days of use.  
- **Customer Sentiment:** Negative.  
- **Next Action:** Escalate to the hardware repair team, initiate a replacement request, and schedule a return shipment for the defective unit.
